In [ ]:
# This notebook is intended to analyse if we can use the IBM power 9
# machine to execute the experiment.

# conclusion: we can.

2

In [56]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm

GRANURALITIES = ["file", "class", "method", "block"]
DATA_FOLDER_PATH = "./code.snippets-with-labels.metrics/"

code_snippets = {}

# Iterates over the directory and prints only files
for granularity in tqdm(GRANURALITIES):
    folder_path = f"{DATA_FOLDER_PATH}/{granularity}"
    path = Path(folder_path)
    projects_file_names = [f.name for f in path.iterdir() if f.is_file()]

    code_snippets[granularity] = {}
    for project_file_name in projects_file_names:
        project_name = project_file_name.split("-")[0]
        
        code_snippets[granularity][project_name] = pd.read_csv(f"{folder_path}/{project_file_name}")
    

PROJECTS_NAMES = list(code_snippets[GRANURALITIES[0]].keys())

LABEL_COLUMN_NAME = "CommentsAssociatedLabel"

100%|██████████| 4/4 [00:07<00:00,  1.98s/it]


In [ ]:
from openai import OpenAI

# Point to your local forwarded SSH port
client = OpenAI(base_url="http://localhost:8085/v1/", api_key="token-if-needed")

def generate_prompt(code_snippet):
    return f"""Given the following code snippet, tell "yes" if it contains a Technical Debt or "no" if it does not contain a Technical Debt. 
You only can answer "yes" or "no". Do not provide any explanation. 

Here is the code snippet: \"
{code_snippet}

\"
"""


def generate(code_snippet):
    prompt = generate_prompt(code_snippet)
    completion = client.chat.completions.create(
        model="/home/multiarq/.cache/instructlab/models/ibm-granite/granite-3.1-8b-instruct",
        messages=[
            {"role": "system", "content": prompt}
        ],
        max_tokens=2,
    )

    return completion.choices[0].message.content.strip()

file_df = code_snippets["block"][PROJECTS_NAMES[0]]
results = []
for i in tqdm(range(400)):
    example = file_df.Content[i]
    try:
        result = generate(example)
    except OpenAI.BadRequestError as e:
        print(f"\nSkipping index {i} due to BadRequestError: {e}")
        results.append(None)  # Keeps alignment with your DataFrame indices
        continue
    results.append(result)


100%|██████████| 400/400 [02:22<00:00,  2.81it/s]


In [ ]:
# file level:
#    for 400 examples, it took 3 minutes and 16 seconds (196 seconds)
# same thing for class metho

# pratically the same thing for everyone
# block: 400 in 2m 22sec

# it will take over 40 hours of experiment.

In [ ]:
numeric_results = [1 if r.lower().startswith("yes") else 0 for r in results]

labels = file_df[LABEL_COLUMN_NAME][:1000].tolist()

In [86]:
for num in numeric_results + labels:
    if type(num) != int:
        print("Non-integer value found:", num)

In [88]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Replace these lists with your actual data
# ground_truth = [1, 0, 1, 1, 0, 1, 0, 0, 1, 0]
# predictions  = [1, 0, 1, 0, 0, 1, 1, 0, 1, 0]

ground_truth = labels
predictions = numeric_results

# Calculate metrics
precision = precision_score(ground_truth, predictions)
recall = recall_score(ground_truth, predictions)
f1 = f1_score(ground_truth, predictions)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Precision: 0.3678
Recall:    0.8649
F1 Score:  0.5161


In [61]:
labels

[0, 1, 0, 1, 1, 1, 1, 1, 1, 0]